<a href="https://colab.research.google.com/github/madhum321/madhumitha-portfolio/blob/main/AISummaryNewsAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq gradio langchain langchain-groq langchain-community langchain-huggingface langchain-text-splitters pypdf faiss-cpu sentence-transformers

In [ ]:
import groq
import gradio
import langchain
import pypdf
import faiss
import sentence_transformers

print("All packages installed successfully!")

All packages installed successfully!


In [ ]:
import os
import gradio as gr

from getpass import getpass

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool

/tmp/ipykernel_1628/451773694.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

Enter your Groq API key: ··········


In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2
)

print("Groq connected successfully!")

Groq connected successfully!


In [ ]:
response = llm.invoke(
    "Explain Artificial Intelligence in two simple sentences."
)

print(response.content)

Artificial Intelligence (AI) is a field of computer science that creates programs capable of learning, reasoning, and making decisions like a human would. It lets machines recognize patterns, understand language, and solve problems without being explicitly programmed for every specific task.


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded))

print("Uploaded PDF:", pdf_path)

Saving sample_ai_news.pdf to sample_ai_news.pdf
Uploaded PDF: sample_ai_news.pdf


In [ ]:
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 2


In [ ]:
print(documents[0].page_content[:1000])

Sample AI News Collection
For AI News Summary Agent Testing
Article 1: AI Assistants Become More Useful for Everyday Tasks
Date: October 1, 2026
Category: Artificial Intelligence
Source: Sample Technology News
AI assistants are increasingly being used for everyday activities such as writing, learning, research,
coding, and information search. Newer systems can use external tools to obtain current information and
can complete several steps before presenting an answer. Developers are also focusing on making these
systems easier to control and more transparent about their sources.
Article 2: Agentic AI Systems Gain Attention
Date: September 29, 2026
Category: Agentic AI
Source: Sample AI Weekly
Agentic AI refers to AI systems that can perform a sequence of tasks toward a goal. Instead of only
generating text, an agent may decide when to use a search tool, process information, call another service,
and then produce a final result. Educational projects are using simple agents for research, 

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [19]:
print(chunks[0].page_content)

Sample AI News Collection
For AI News Summary Agent Testing
Article 1: AI Assistants Become More Useful for Everyday Tasks
Date: October 1, 2026
Category: Artificial Intelligence
Source: Sample Technology News
AI assistants are increasingly being used for everyday activities such as writing, learning, research,
coding, and information search. Newer systems can use external tools to obtain current information and
can complete several steps before presenting an answer. Developers are also focusing on making these
systems easier to control and more transparent about their sources.
Article 2: Agentic AI Systems Gain Attention
Date: September 29, 2026
Category: Agentic AI
Source: Sample AI Weekly
Agentic AI refers to AI systems that can perform a sequence of tasks toward a goal. Instead of only
generating text, an agent may decide when to use a search tool, process information, call another service,


In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Hugging Face embeddings created successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Hugging Face embeddings created successfully!


In [21]:
vector_db = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector database created successfully!")

Vector database created successfully!


In [22]:
vector_db.save_local("news_faiss_index")

In [23]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created successfully!")

Retriever created successfully!


In [24]:
question = "What are the main developments discussed in the news?"

retrieved_docs = retriever.invoke(question)

print("Number of retrieved chunks:", len(retrieved_docs))

Number of retrieved chunks: 4


In [25]:
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content[:1000])


--- Retrieved Chunk 1 ---
Upload this PDF into the AI News Summary Agent. The agent should extract the article text and generate
a short overall summary, important key points, and article-wise summaries. This document is synthetic
sample content created for project testing and is not a collection of real news reports.

--- Retrieved Chunk 2 ---
Sample AI News Collection
For AI News Summary Agent Testing
Article 1: AI Assistants Become More Useful for Everyday Tasks
Date: October 1, 2026
Category: Artificial Intelligence
Source: Sample Technology News
AI assistants are increasingly being used for everyday activities such as writing, learning, research,
coding, and information search. Newer systems can use external tools to obtain current information and
can complete several steps before presenting an answer. Developers are also focusing on making these
systems easier to control and more transparent about their sources.
Article 2: Agentic AI Systems Gain Attention
Date: September 29, 20

In [26]:
@tool
def rag_tool(question: str) -> str:
    """
    Search the uploaded news PDF and return relevant information.
    """

    docs = retriever.invoke(question)

    if not docs:
        return "No relevant information was found in the PDF."

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    return context

In [27]:
result = rag_tool.invoke(
    "Summarize the main news topics."
)

print(result)

Source: Sample Digital News
News summarization systems can process multiple articles and present the main information in a shorter
format. A typical workflow includes collecting articles, filtering relevant content, removing duplicate
information, extracting important facts, and generating a concise summary. Showing the original source
with the summary helps users check the information.
Testing Task

Upload this PDF into the AI News Summary Agent. The agent should extract the article text and generate
a short overall summary, important key points, and article-wise summaries. This document is synthetic
sample content created for project testing and is not a collection of real news reports.

generating text, an agent may decide when to use a search tool, process information, call another service,
and then produce a final result. Educational projects are using simple agents for research, document
analysis, scheduling, and news summarization.
Article 3: AI Tools Support Students and Teache

In [28]:
@tool
def budget_tool(
    income: float,
    expenses: float
) -> str:
    """
    Calculate remaining budget and spending percentage.
    """

    if income <= 0:
        return "Income must be greater than zero."

    remaining = income - expenses

    percentage = (expenses / income) * 100

    return (
        f"Income: ₹{income:.2f}\n"
        f"Expenses: ₹{expenses:.2f}\n"
        f"Remaining Budget: ₹{remaining:.2f}\n"
        f"Spending Percentage: {percentage:.2f}%"
    )

In [29]:
result = budget_tool.invoke({
    "income": 50000,
    "expenses": 32000
})

print(result)

Income: ₹50000.00
Expenses: ₹32000.00
Remaining Budget: ₹18000.00
Spending Percentage: 64.00%


In [30]:
tools = [
    rag_tool,
    budget_tool
]

llm_with_tools = llm.bind_tools(tools)

In [31]:
tool_map = {
    "rag_tool": rag_tool,
    "budget_tool": budget_tool
}

In [32]:
def run_agent(user_question):

    messages = [
        (
            "system",
            """
You are an Agentic AI News Summary Agent.

Your responsibilities:

1. Use rag_tool when the user asks about
   information contained in the uploaded PDF.

2. Use budget_tool when the user asks for
   budget or financial calculations.

3. If a tool is not required, answer directly.

4. Do not invent information from the PDF.

5. Give clear and simple answers.
            """
        ),
        ("human", user_question)
    ]

    for _ in range(4):

        response = llm_with_tools.invoke(messages)

        if not response.tool_calls:
            return response.content

        messages.append(response)

        for call in response.tool_calls:

            tool_name = call["name"]
            tool_args = call["args"]

            result = tool_map[tool_name].invoke(tool_args)

            messages.append({
                "role": "tool",
                "content": str(result),
                "tool_call_id": call["id"]
            })

    return "The agent reached its tool-call limit."

In [33]:
answer = run_agent(
    "What are the main topics discussed in the uploaded news PDF?"
)

print(answer)

The PDF covers four primary topics:

1. **AI assistants for everyday tasks** – how they’re being used for writing, learning, research, coding, and information search, and the push for better control and source transparency.  
2. **Agentic AI systems** – AI that can plan and execute multi‑step tasks, decide when to call tools or services, and examples of educational projects that use simple agents.  
3. **AI tools in education** – applications such as generating study summaries, explaining concepts, creating practice questions, and organizing resources, along with concerns about privacy, accuracy, and responsible use.  
4. **News‑summarization technology** – the workflow for collecting, filtering, de‑duplicating, extracting facts, and producing concise summaries, and how this helps users save time.


In [34]:
answer = run_agent(
    "My income is 60000 and my expenses are 42000. "
    "How much money do I have remaining?"
)

print(answer)

Your remaining budget is **₹18,000**, which means you’ve spent **70 %** of your income.


In [35]:
def chat(question):

    if not question.strip():
        return "Please enter a question."

    try:
        return run_agent(question)

    except Exception as e:
        return f"Error: {e}"

In [36]:
with gr.Blocks(
    title="AI News Summary Agent"
) as demo:

    gr.Markdown(
        "# 🤖 Agentic AI News Summary Agent"
    )

    gr.Markdown(
        """
        Ask questions about your uploaded news PDF
        or use the budget calculation tool.
        """
    )

    question = gr.Textbox(
        label="Enter Your Question",
        placeholder="Ask about the news..."
    )

    output = gr.Textbox(
        label="AI Agent Response",
        lines=12
    )

    button = gr.Button("Ask Agent")

    button.click(
        fn=chat,
        inputs=question,
        outputs=output
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc9a72b1e0a6184b63.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
